# Pipeline stage 3: characterization of phage-encoded acetyltransferase Map

### Stage 3.2: conservation of Map across the *Phikmvvirus* genus

In our research, we have characterized LUZ19 gp13. LUZ19 is a *Phikmvvirus*, and as we have shown in our initial characterization of the candidate acetyltransferases, many *Phikmvviruses* encode both a class Ia acetyltransferase (such as LUZ19 gp13) and a class IIb acetyltransferase (such as LUZ19 Rac). However, we did not find a type Ia acetyltransferase in all *Phikmvviruses* we considered, so let's see if this is due to technical or biological reasons, by performing a complentary, DNA-based sequence search for homologs of LUZ19 gp13 across *Phikmvviruses*.

**Goals:**
* Obtain genome sequences for all *Phikmvviruses* in our dataset.
* Prepare & analyze MMSeqs2 sequence search.
* Cross-link DNA-based alignment to protein annotations.
* Visualize the conservation of the Map protein in *Phikmvviruses*.

**Requires:**
* phage_data.tsv stored in 1_search/a_input : a tab-seperated phage-centric overview of the virus & host taxonomic information from the Virus-Host database, summarizing all included phages (one phage per line).
* virushostdb_host_pseudomonas_25feb2024.tsv stored in associated_data : results of search for 'Pseudomonas' as host in the Virus-Host database (version: February 25th, 2024).
* predicted_acetyltransferase_overview.tsv stored in 2_characterization/a_overview : a tab-seperated overview of the final selection of proteins of our phages that are predicted to be acetyltranferases. Each row represents one unique phage-acetyltransferase pair.

**Generates:**
* in folder align:
    * phikmvvirus.fasta : a multi-FASTA file containing genome sequence information from all *Pseudomonas*-infecting *Phikmvviruses* as listed by the Virus-Host database, fasta headers represent the name_taxid used throughout the code to refer to specific phages.
    * map.fasta : a FASTA file containing the protein sequence of Map, as described by Lavigne *et al* (2013).
    * mmseqs.pbs : the .pbs job script to carry out the protein-vs-6-frame translation search with MMSeqs
    * MMseqs output files: 
        * map(.XX) with XX indicating a dbtype, index, lookup or source file
        * map_h(.XX) with XX indicating a dbtype, index
        * map_homologs.XX with XX indicting a tsv, index or dbtype file
        * phikmvviruses(.XX) with XX indicating a dbtype, index, lookup or source file
        * phikmvviruses_h(.XX) with XX indicating a dbtype, index
        * MMseqs.e/o file: error/output files from HPC job
* in folder compare:
    * PHAGE.gb : GenBank formatted genome annotation files for every *Phikmvvirus* PHAGE 
* in folder investigate_early_starts:
    * in folder expasy_translate:
        * PHAGE_nt_START-STOP_nuccore_ID.fasta : input used for ExPASy translate, for PHAGE, with nucleotides taken from NCBI nuccore with accession ID, from residue START up until residue STOP
        * PHAGE_nt_START-STOP_nuccore_ID_translate.txt : ExPASy translate output of selected in-frame elongated variant
    * clustalo-I20260210-150327-0134-7269779-p2m.aln-clustal_num : Clustal Omega alignment file obtained by running Clustal O through the EBI webserver on the extended (initially truncated) Map homologs, using all default settings.    
* in folder lovis4u:
    * in folder gb:
        * PHAGE_modifed.gb : modified GenBank formatted genome annotation files for every *Phikmvvirus* PHAGE. The original GenBank files were modified (see code for more details) to make Map homologs easily identifiable, and to correct start sites for identified truncated Map homologs.   
    * lovis4u_full.slurm : the .slurm job script to run LoVis4u on the complete *Phikmvvirus* genomes
    * in folder full_genomes:  
        *  lovis4u output:
            * feature_annotation_table.tsv, locus_annotation_table.tsv, proteome_similarity_matrix.tsv and lovis4u.pdf   
            * folder mmseqs:
                * mmseqs_clustering.tsv, input_proteins.fa, mmseqs_stderr.txt, mmseqs_stoud.txt
                * folder DB:
                    * clusterDB.XX with XX a number ranging from 00 to 71
                    * clusterDB.XX with XX indicating a index or dbtype file
                    * sequencesDB(.XX) with XX indicating a dbtype, index, lookup or source file
                    * sequencesDB_h(.XX) with XX indicating a dbtype, index
    * lovis4u_map.slurm : the .slurm job script to run LoVis4u on the early genomic region of *Phikmvvirus* genomes
    * lat.tsv : the .tsv formatted locus annotation table passed to LoVis4u encoding the genomic regions corresponing to phiKMV gp1-gp19
    * in folder map_region: the exact same as the full_genomes folder, only now visualizing a restricted genomic region instead of the full phage genomes


#### General settings, imports, variables and environments

Conda environment: viral_act_gp13_cons

Created with `conda create -n viral_act_gp13_cons python=3.10`. Then installed `jupyter notebook`, `pandas` through conda (command `conda install`) & `biopython` through pip (command below).

In [ ]:
!pip install biopython

In [1]:
!conda list --explicit

# This file may be used to create an environment using:
# $ conda create --name <env> --file <this file>
# platform: win-64
# created-by: conda 24.11.3
@EXPLICIT
https://conda.anaconda.org/conda-forge/noarch/ca-certificates-2026.1.4-h4c7d964_0.conda
https://conda.anaconda.org/conda-forge/noarch/python_abi-3.10-8_cp310.conda
https://conda.anaconda.org/conda-forge/noarch/tzdata-2025c-hc9c84f9_1.conda
https://conda.anaconda.org/conda-forge/win-64/ucrt-10.0.26100.0-h57928b3_0.conda
https://conda.anaconda.org/conda-forge/win-64/winpty-0.4.3-4.tar.bz2
https://conda.anaconda.org/conda-forge/win-64/libwinpthread-12.0.0.r4.gg4f2fc60ca-h57928b3_10.conda
https://conda.anaconda.org/conda-forge/win-64/vcomp14-14.44.35208-h818238b_34.conda
https://conda.anaconda.org/conda-forge/win-64/vc14_runtime-14.44.35208-h818238b_34.conda
https://conda.anaconda.org/conda-forge/win-64/vc-14.3-h41ae7f8_34.conda
https://conda.anaconda.org/conda-forge/win-64/bzip2-1.0.8-h0ad9c76_8.conda
https://conda.anaconda.org/c

In [2]:
# imports 
import os
import requests

import pandas as pd

from Bio import SeqIO
from Bio.SeqFeature import FeatureLocation

In [3]:
# settings for requests
sess = requests.Session()
adapter = requests.adapters.HTTPAdapter(max_retries = 10)
sess.mount("https://", adapter)

In [4]:
# clear reference to different directories
pipeline_map_dir = os.getcwd()
master_dir = os.path.abspath(os.path.join(pipeline_map_dir, os.pardir, os.pardir))
pipeline_search_dir = os.path.join(master_dir, "pipeline", "1_search")
pipeline_char_dir = os.path.join(master_dir, "pipeline", "2_characterization")

In [ ]:
# creating a directory for all the data we will generate
os.mkdir(os.path.join(pipeline_map_dir, "b_conservation"))

#### Goal 1 : obtain genome sequences for all *Phikmvviruses* in our dataset

To start the process of searching for Map homolog conservation across *Phikmvviruses*, we first extract all phages in this genus:

In [5]:
# reading in the data
phage_data = pd.read_csv(os.path.join(pipeline_search_dir, "a_input", "phage_data.tsv"), sep = "\t")

In [6]:
# filtering for Phikmvvirus
phikmv_data = phage_data[phage_data["virus_lineage"].str.contains("Phikmv")]
print(f"Our dataset contains {phikmv_data['virus_taxid'].nunique()} different Phikmvviruses.")

Our dataset contains 41 different Phikmvviruses.


Next, we obtain their genome information from NCBI, based on the RefSeq identifiers we originally obtained from the Virus-Host database:

In [7]:
# reading in the data
virushost_data = pd.read_table(os.path.join(master_dir, "associated_data", "virushostdb_host_pseudomonas_25feb2024.tsv"))

In [8]:
# obtaining the RefSeq identifiers
phikmv_data["refseq_id"] = phikmv_data.apply(lambda row : virushost_data[virushost_data["virus tax id"] == row["virus_taxid"]]["refseq id"].iloc[0], axis = 1)

C:\Users\hanne\AppData\Local\Temp\ipykernel_10804\4201443992.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  phikmv_data["refseq_id"] = phikmv_data.apply(lambda row : virushost_data[virushost_data["virus tax id"] == row["virus_taxid"]]["refseq id"].iloc[0], axis = 1)


Using those RefSeq identifiers, we obtain the full genome sequences from NCBI.

In [9]:
# function to fetch the genome sequences from NCBI
def fetch_sequence_ncbi(refseq_id):
    url_fasta = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=nuccore&id={0}&rettype=fasta"
    # getting info
    fasta = sess.get(url_fasta.format(refseq_id), stream = True).text
    seq = fasta.split("\n")
    return "".join(seq[1:-2])

In [10]:
# obtaining the genome sequences
phikmv_data["genome_seq"] = phikmv_data.apply(lambda row : fetch_sequence_ncbi(row["refseq_id"]), axis = 1)

C:\Users\hanne\AppData\Local\Temp\ipykernel_10804\3845814124.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  phikmv_data["genome_seq"] = phikmv_data.apply(lambda row : fetch_sequence_ncbi(row["refseq_id"]), axis = 1)


#### Goal 2 : prepare & analyze MMSeqs2 sequence search

For our sequence search, we need a fasta file containing all our search sequences and a fasta file containing our query, which we generate here. 

In [ ]:
# directory for this alignment analysis 
os.mkdir(os.path.join(pipeline_map_dir, "b_conservation", "align"))

In [ ]:
# input fasta file with Phikmvvirus genomes
mmseqs_input_gen = os.path.join(pipeline_map_dir, "b_conservation", "align", "phikmvvirus.fasta")
with open(mmseqs_input_gen, "w") as file:
    for index, row in phikmv_data.iterrows():
        name_taxid = row["name_taxid"]
        sequence = row["genome_seq"]
        file.write(f">{name_taxid}\n")
        file.write(f"{sequence}\n")

In [12]:
# reading in the data storing LUZ19 Map
predicted_act = pd.read_csv(os.path.join(pipeline_char_dir, "a_overview", "predicted_acetyltransferase_overview.tsv"), sep = "\t")

In [ ]:
# input fasta file with LUZ19 Map
mmseqs_input_prot = os.path.join(pipeline_map_dir, "b_conservation", "align", "map.fasta")
with open(mmseqs_input_prot, "w") as file:
    luz19_map = predicted_act[predicted_act["identification_method"] == "manual addition"]
    sequence = luz19_map["sequence"].values[0]
    file.write(f">LUZ19_Map\n")
    file.write(f"{sequence}\n")

Next, we create a job script that will perform a protein search against a six frame translated genome (search-type 2), with strict E-value cut-off to ensure homology.

In [ ]:
script_content = '''#!/bin/bash
#PBS -N MMseqs
#PBS -l nodes=1:ppn=1
#PBS -l walltime=1:00:00

cd $PBS_O_WORKDIR
export OMP_PROC_BIND=false

module load MMseqs2/14-7e284-gompi-2023a

mmseqs createdb phikmvvirus.fasta phikmvviruses
mmseqs createdb map.fasta map

mmseqs search map phikmvviruses map_homologs $TMPDIR --search-type 2 -e 1e-50
mmseqs convertalis map phikmvviruses map_homologs map_homologs.tsv --format-output "query,target,fident,alnlen,mismatch,qstart,qend,qcov,tstart,tend,tcov,evalue,bits"
'''

In [ ]:
script = os.path.join(pipeline_map_dir, "b_conservation", "align", "mmseqs.pbs")
with open(script, "w") as file:
    print(script_content, file = file)

Using the created scripts, we searched for Map homologs across all *Phikmvviruses* considered in this study. Let's have a look at the results.

In [13]:
# reading in the output
mmseqs_out = pd.read_csv(os.path.join(pipeline_map_dir, "b_conservation", "align", "map_homologs.tsv"), sep = "\t", header = None,
                        names = ["query", "target", "fident", "alnlen", "mismatch", "qstart", "qend", "qcov",
                                 "tstart", "tend", "tcov", "evalue", "bits"])
# checking the number of phages with Map homolog
if phikmv_data["virus_taxid"].nunique() == mmseqs_out["target"].nunique():
    print("All Phikmvviruses have an identifiable homolog of Map.")
    if len(phikmv_data) == len(mmseqs_out):
        print("All Phikmvviruses have exactly one identifiable homolog of Map.")
    else:
        print("The number of MMseqs hits is different from the number of Phikmvviruses, pointing to multiple hits in the same phage.")

All Phikmvviruses have an identifiable homolog of Map.
The number of MMseqs hits is different from the number of Phikmvviruses, pointing to multiple hits in the same phage.


In [14]:
# let's look into the phage(s) with multiple hits
    # identify those with multiple hits
mmseqs_out_phage_count = mmseqs_out.groupby("target").size().reset_index(name = "count")
multiple_hits = list(mmseqs_out_phage_count[mmseqs_out_phage_count["count"] > 1]["target"])
    # print dataframe with only those
mmseqs_out[mmseqs_out["target"].isin(multiple_hits)]

,query,target,fident,alnlen,mismatch,qstart,qend,qcov,tstart,tend,tcov,evalue,bits
40,LUZ19_Map,vB_PaeP_FBPa18_taxid_2969611,0.987,288,1,1,96,0.533,6912,7199,0.006,3.223000e-60,202
41,LUZ19_Map,vB_PaeP_FBPa18_taxid_2969611,0.900,279,9,87,179,0.517,7208,7486,0.006,1.848000e-51,177


As we can see, **we actually do identify a Map homolog in each *Phikmvvirus***, on the basis of this MMseqs search. As such, the fact that we initially did not find a type I acetyltransferase in every *Phikmvvirus* is likely due to technical reasons, rather than biological reasons. Let's see if we can identify those. For phage FBPa18, we observe two partial hits in the MMseqs results, very closely together, which could perhaps be due to an assembly issue.

#### Goal 3: cross-link DNA-based alignment to protein annotations

Let's investigate why we get hits for each of the *Phikmvviruses* based on an MMSeqs search, when they were not all detected by our pipeline.

The reasons could be multiple:
* protein annotations being different from 'protein of unknown function'/'acetyltransferase' (i.e. a known function that is not an acetyltransferase)
* gene calling issues 
* for the partail hits: proteins being too small (proteins of unknown function were not considered if smaller than 100 amino acids) 

Let's start by looking into gene calling/protein annotations, by checking whether the MMSeqs match on the DNA level corresponds to an actually called gene, and how that gene was annotated. To do so, we need the GenBank files, so let's obtain those first.

In [ ]:
# directory to store this information
os.mkdir(os.path.join(pipeline_map_dir, "b_conservation", "compare"))

In [16]:
# function to fetch the GenBank files from NCBI
def fetch_gb_ncbi(id, output_loc):
    url_gb = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=nucleotide&id={0}&rettype=gb&retmode=text"
    # getting info
    gb = sess.get(url_gb.format(id), stream = True).text
    # storing to file
    with open(output_loc, "w") as outfile:
        outfile.write(gb)

In [ ]:
# apply
for index, row in phikmv_data.iterrows():
    refseq = row["refseq_id"]
    name_taxid = row["name_taxid"]
    fetch_gb_ncbi(refseq, os.path.join(pipeline_map_dir, "b_conservation", "compare", f"{name_taxid}.gb"))

Now that we have obtained those files, we can compare the genomic coordinates of our MMseqs matches to protein coordinates in the GFF files. Let's first identify all identical matches:

In [17]:
# let's first see if we can find exact matches
    # list of exact matches
phage_protein_match = list()
phage_no_match = list()
    # loop through all Phikmvviruses to find matches
for index, row in mmseqs_out.iterrows():
    # extract match properties
    name_taxid = row["target"]
    tstart = int(row["tstart"])
    tstop = int(row["tend"])
    match = False
    if tstart < tstop:
        aln_start = tstart - 1 # correct for 0-based indexing of start
        aln_stop = tstop + 3 # correct for stop codon
    # correct for reverse strand
    else: 
        aln_start = tstop - 4 # correct for stop codon + 0-based indexing of start
        aln_stop = tstart  
    # read in genbank record
    record = SeqIO.read(os.path.join(pipeline_map_dir, "b_conservation", "compare", f"{name_taxid}.gb"), "genbank")
    # find match
    for feature in record.features:
        if feature.type == "CDS":
            if (int(feature.location.start) == aln_start) and (int(feature.location.end) == aln_stop):
                phage_protein_match.append((name_taxid, feature.qualifiers.get("product")[0], feature.qualifiers.get("translation")[0]))
                match = True
    if match == False:
        phage_no_match.append((name_taxid, tstart, tstop, aln_start, aln_stop))
print(f"There are {len(phage_protein_match)} phages that have a match (Map homolog) on the DNA level, and a gene called at exactly these match coordinates.")
print("Hence, these proteins should have been detected by our pipeline.")

There are 36 phages that have a match (Map homolog) on the DNA level, and a gene called at exactly these match coordinates.
Hence, these proteins should have been detected by our pipeline.


Let's look into the identical matches first: did we capture them in our phage acetyltransferase search?

In [18]:
# count the number we did not find in our pipeline
count_not_found = 0
annotat_not_found = list()
# loop over matches
for entry in phage_protein_match:
    # extract relevant properties
    name_taxid = entry[0]
    annotation = entry[1]
    sequence = entry[2]
    # check if in acetyltransferase data
    match = predicted_act[(predicted_act["name_taxid"] == name_taxid) & (predicted_act["sequence"] == sequence)]
    if len(match) == 0:
        count_not_found += 1
        annotat_not_found.append(annotation)
print(f"In total, {count_not_found} proteins were not detected by our pipeline, these were annotated as: {set(annotat_not_found)}.")

In total, 7 proteins were not detected by our pipeline, these were annotated as: {'DNA primase', 'putative DNA primase'}.


As we can see, of the 36 exact matches, we captured 29. **The 7 we did not capture had never been subjected to our pipeline, as they are annotated as (putative) DNA primase**. As such, this is not a mistake by our pipeline, but simply a technical consequence of our filtering (we only considered proteins of unknown function or annotated acetyltransferases).

Let's now take a look at those where MMSeqs identified a DNA level match, but where there is no gene called at the exact match coordinates.

In [19]:
# return list to get an overview of the phages and the coordinates of the matches
phage_no_match

[('LUZ19_taxid_484896', 7021, 7560, 7020, 7563),
 ('DL62_taxid_1640972', 6885, 7424, 6884, 7427),
 ('MPK6_taxid_1262514', 7050, 7586, 7049, 7589),
 ('JB10_taxid_3028140', 7005, 7541, 7004, 7544),
 ('vB_PaeP_FBPa18_taxid_2969611', 6912, 7199, 6911, 7202),
 ('vB_PaeP_FBPa18_taxid_2969611', 7208, 7486, 7207, 7489)]

Interestingly, we see that the only phages for which we do not get a protein at the exact start/stop of the DNA level match to Map, are:
* vB_PaeP_FBPa18, for which we previously discussed the fact that we got two (partial) matches, which could potentially be due to misassembly
* the phages for which we identified potential earlier start codons (LUZ19, MPK6) and/or a class If acetyltransferase (MPK6, DL62, JB10)

Given the full length match at a DNA level (for all but phage FBPa18, but there we have two ~50% matches), it is possible that all these phages actually contain full length Map homologs, for which an earlier start codon is available, but only a shorter, in-frame variant has been gene called. Let's check whether there are indeed shorter, annotated, in-frame protein annotations for those matches, that start later.

In [21]:
partial_protein_match = list()
for phage_match in phage_no_match:
    name_taxid = phage_match[0]
    aln_start = phage_match[3]
    aln_stop = phage_match[4]
    # read in genbank record
    record = SeqIO.read(os.path.join(pipeline_map_dir, "b_conservation", "compare", f"{name_taxid}.gb"), "genbank")
    # find match
    for feature in record.features:
        if feature.type == "CDS":
            if (int(feature.location.end) == aln_stop) and (int(feature.location.start) > aln_start) and ((int(feature.location.start)-aln_start)%3 == 0):
                print(f"Found in-frame, shorter protein variant of Map for phage {name_taxid}.")
                partial_protein_match.append((name_taxid, feature.qualifiers.get("product")[0], feature.qualifiers.get("translation")[0]))

Found in-frame, shorter protein variant of Map for phage LUZ19_taxid_484896.
Found in-frame, shorter protein variant of Map for phage DL62_taxid_1640972.
Found in-frame, shorter protein variant of Map for phage MPK6_taxid_1262514.
Found in-frame, shorter protein variant of Map for phage JB10_taxid_3028140.
Found in-frame, shorter protein variant of Map for phage vB_PaeP_FBPa18_taxid_2969611.


Indeed, for all these cases, an in-frame, shorter, annotated protein has been gene called. 

Let's also check whether these shorter proteins were detected by our pipeline:

In [22]:
# count the number we did not find in our pipeline
count_not_found_p = 0
annotat_not_found_p = list()
# loop over matches
for entry in partial_protein_match:
    # extract relevant properties
    name_taxid = entry[0]
    annotation = entry[1]
    sequence = entry[2]
    # check if in acetyltransferase data
    match = predicted_act[(predicted_act["name_taxid"] == name_taxid) & (predicted_act["sequence"] == sequence)]
    if len(match) == 0:
        if (len(sequence) < 100) & (annotation == "hypothetical protein"):
            print(f"The match for phage {name_taxid} was not found by our pipeline. However, as it was a protein of unknown function smaller than 100 amino acids, it was never considered in the first place.")
        else:
            count_not_found_p += 1
            annotat_not_found_p.append(annotation)
print(f"In total, {count_not_found_p} proteins were not detected by our pipeline, these were annotated as: {set(annotat_not_found_p)}.")

The match for phage LUZ19_taxid_484896 was not found by our pipeline. However, as it was a protein of unknown function smaller than 100 amino acids, it was never considered in the first place.
The match for phage vB_PaeP_FBPa18_taxid_2969611 was not found by our pipeline. However, as it was a protein of unknown function smaller than 100 amino acids, it was never considered in the first place.
In total, 0 proteins were not detected by our pipeline, these were annotated as: set().


As we can see, **all but two proteins were actually labeled as acetyltransferases by our pipeline. The two that are not labeled as acetyltransferases, were never considered by the pipeline as they are proteins of unknown function smaller than 100 amino acids**.

Let's now finally test the hypothesis that these cases, where there is a full length DNA match to Map, but no corresponding full length protein match, actually represent proteins which could be elongated to Map homologs. More concretely, let's check whether there is an upstream, in-frame start codon (and no stop codon inbetween). For LUZ19, we know that the NCBI protein annotation is incomplete (Lavigne *et al*, 2013). For MPK6, we have previously shown that the protein could be extended into a full length Map homolog (notebook Code_search_assess_annotated.ipynb). Let's use the same analysis now for DL62, JB10 and FBPa18. 

The input (a wide range of DNA sequence around the encoded acetyltransferase) is stored in investigate_early_start/expasy_translate. EXPASY translate is used with 'standard genetic code' and 'Forward' and 'Reverse' strand selected. The output is stored in the file translate.txt.

Based on the EXPASY results, we can identify full length Map homologs in DL62 and JB10, i.e., EXPASY does suggest an in-frame translation with earlier start codon. However, for FBPa18, this is not the case. Given our previous results (partial matches to two different fractions, close by in the genome), and the otherwise complete conservation of Map across the entire *Phikmvvirus*-genus, it is possible that this is due to an assembly-level issue.

#### Goal 4: visualize the conservation of the Map protein in Phikmvviruses

Let's now visualize the conservation, using Lovis4u. To do this, we will create 'corrected' GenBank files that contain the correct 'Map' protein coordinates, and remove any mis-/truncated annotations, which we will then visualize. To make the visualization clear, we will not visualize the whole genomes, but instead zoom into the Map-surrounding region. To do so, we will first correct the GenBank files. 

In [ ]:
# directory to store this information
os.mkdir(os.path.join(pipeline_map_dir, "b_conservation", "lovis4u"))
os.mkdir(os.path.join(pipeline_map_dir, "b_conservation", "lovis4u", "gb"))

For the matches where a protein has been called at the exact match coordinates, we will just update the protein annotation to 'Map', to make these proteins easily identifiable.

In [ ]:
# start with the easy case, those with Map correctly annotated
for entry in phage_protein_match:
    name_taxid = entry[0]
    tln = entry[2]
    # read in genbank record
    record = SeqIO.read(os.path.join(pipeline_map_dir, "b_conservation", "compare", f"{name_taxid}.gb"), "genbank")
    # find match
    for feature in record.features:
        if feature.type == "CDS" and feature.qualifiers.get("translation")[0] == tln:
            # update the protein annotation
            feature.qualifiers["product"] = ["Map"]
    # write modified genbank file
    out_path = os.path.join(pipeline_map_dir, "b_conservation", "lovis4u", "gb", f"{name_taxid}_modified.gb")
    SeqIO.write(record, out_path, "genbank")


For the matches where we believe the gene caller has missed an earlier start codon that could elongate the protein into a full length Map homolog, we will correct the CDS coordinates, the protein sequence and the protein annotation.

In [ ]:
# now for the partial matches
for phage_match in phage_no_match:
    name_taxid = phage_match[0]
    aln_start = phage_match[3]
    aln_stop = phage_match[4]
    if "FBPa18" in name_taxid:
        continue # let's deal with this one later, has two partial matches 
    elif "LUZ19" in name_taxid:
        tln = predicted_act[predicted_act["identification_method"] == "manual addition"]["sequence"].values[0]
    elif "MPK6" in name_taxid:
        tln = predicted_act[(predicted_act["name_taxid"] == name_taxid) & (predicted_act["elongated"] == True)]["sequence"].values[0]
    else:
        matches = [f for f in os.listdir(os.path.join(pipeline_map_dir, "b_conservation", "investigate_early_starts", "expasy_translate")) if (name_taxid in f) and ("translate" in f)]
        fasta = SeqIO.read(os.path.join(pipeline_map_dir, "b_conservation", "investigate_early_starts", "expasy_translate", matches[0]), "fasta")
        tln = str(fasta.seq)
    # read in genbank record
    record = SeqIO.read(os.path.join(pipeline_map_dir, "b_conservation", "compare", f"{name_taxid}.gb"), "genbank")
    # find match
    for feature in record.features:
        if feature.type == "CDS":
            if (int(feature.location.end) == aln_stop) and (int(feature.location.start) > aln_start) and ((int(feature.location.start)-aln_start)%3 == 0):
                # update the protein annotation
                feature.qualifiers["product"] = ["Map"]
                # update the coordinates
                old_loc = feature.location
                feature.location = FeatureLocation(aln_start, old_loc.end, old_loc.strand)
                # update the translation
                feature.qualifiers["translation"] = [tln]
    # write modified genbank file
    out_path = os.path.join(pipeline_map_dir, "b_conservation", "lovis4u", "gb", f"{name_taxid}_modified.gb")
    SeqIO.write(record, out_path, "genbank")
  

Finally, the case of phage FBPa18 requires some more tinkering. We want to visualize the N- and C-terminal alignments as two different CDSes, just as we found them. For the C-terminal match, this means elongating an existing CDS to an earlier in-frame start codon (like for the partial matches before). For the N-terminal match, the situation is slightly different: here, there is again an in-frame called CDS, but that called CDS goes on beyond our match. To make the visualization as clean as possible, we will show the N-terminal match and truncate its coordinates to stop at the end of the match. 

In [ ]:
# collect both hits first
hits = []
for phage_match in phage_no_match:
    if "FBPa18" in phage_match[0]:
        name_taxid = phage_match[0]
        aln_start = phage_match[3]
        aln_stop  = phage_match[4]
        hits.append((aln_start, aln_stop))
        
# read in genbank record
record = SeqIO.read(os.path.join(pipeline_map_dir, "b_conservation", "compare", f"{name_taxid}.gb"), "genbank")

# Process both hits
for aln_start, aln_stop in hits:
    # find match
    for feature in record.features:
        if feature.type == "CDS":
            # this part deals with the C-terminal match, which is in-frame with a (truncated) annotated CDS
            if (int(feature.location.end) == aln_stop) and (int(feature.location.start) > aln_start) and ((int(feature.location.start)-aln_start)%3 == 0):
                # update the protein annotation
                feature.qualifiers["product"] = ["Map-fragment (C-term)"]
                # update the protein coordinates
                old_loc = feature.location
                feature.location = FeatureLocation(aln_start, old_loc.end, old_loc.strand)
                # update the translation - translate from genome sequence, using the updated coordinates
                feature.qualifiers["translation"] = [str(record.seq[aln_start:aln_stop].translate(to_stop=True)).rstrip("*")]
            # this part deals with the N-terminal match, which aligns with an in-frame annotated CDS, but is shorter    
            elif (int(feature.location.start) == aln_start):
                feature.qualifiers["product"] = ["Map-fragment (N-term)"]
                # update the protein coordinates
                old_loc = feature.location
                feature.location = FeatureLocation(old_loc.start, aln_stop, old_loc.strand)
                # update the translation - translate from genome sequence, using the updated coordinates (but correct for absence stop codon)
                feature.qualifiers["translation"] = [str(record.seq[aln_start:aln_stop-3].translate(to_stop=True)).rstrip("*")]

# write modified genbank file
out_path = os.path.join(pipeline_map_dir, "b_conservation", "lovis4u", "gb", f"{name_taxid}_modified.gb")
SeqIO.write(record, out_path, "genbank")

Let's first visualize the alignment of the full genomes, so we can use this to guide the best region to visualize.

In [ ]:
 # then, we make the HPC script
script_content_full = '''#!/bin/bash

#SBATCH --nodes=1 --ntasks-per-node=8
#SBATCH --time=00:15:00

cd $SLURM_SUBMIT_DIR

source /data/leuven/331/vsc33164/miniconda3/bin/activate
conda activate lovis4u

lovis4u --linux

lovis4u -gb ./gb/ -o full_genomes -rol -hl

'''

In [ ]:
script_full = os.path.join(pipeline_map_dir, "b_conservation", "lovis4u", "lovis4u_full.slurm")
with open(script_full, "w") as file:
    print(script_content_full, file = file)

After inspecting the full genome visualization, we create a locus-annotation-file listing the specific genomic region we want to visualize, zooming in on Map (`lat.tsv` stored in folder `lovis4u`). The coordinates of the genomic region to visualize was extracted manually for each phage, and is meant to encompass Gp1 up until Gp19, in analogy to *Phikmv* (all early to early/middle genes). 

The end coordinate was extracted on the basis of the end coordinate of Gp19. Gp19 was identified as the 'DNA polymerase' following Map in the GenBank annotation files (also annotated as 'DNA polymerase I', '(putative) DNA-directed DNA polymerase', 'putative DNA polymerase A'). 

The start coordinate was extracted as either (i) the first nucleotide of the genome, for genomes following typical *Phikmv* gene ordering, or (ii) the first nucleotide of the most likely Gp1, should genes be reordered to adhere to typical *Phikmv* gene ordering. The most likely Gp1 was assigned based on visual inspection of the full genome alignment results, for phages AIIMS-Pa-A1, phipa2, RLP, sp. LP, vB_PaeA_SB, vB_PaeP_ASP23, vB_PaeP_Lx18, and 20Aug401. 

Finally, we can create a jobscript using this information to run lovis4u and visualize this region.

In [ ]:
 # then, we make the HPC script
script_content_map = '''#!/bin/bash

#SBATCH --nodes=1 --ntasks-per-node=8
#SBATCH --time=00:15:00

cd $SLURM_SUBMIT_DIR

source /data/leuven/331/vsc33164/miniconda3/bin/activate
conda activate lovis4u

lovis4u --linux

lovis4u -gb ./gb/ -o map_region -rol --locus-annotation-file lat.tsv -c A4p1

'''

In [ ]:
script_map = os.path.join(pipeline_map_dir, "b_conservation", "lovis4u", "lovis4u_map.slurm")
with open(script_map, "w") as file:
    print(script_content_map, file = file)

Finally, this visualization was manually post-edited to align all genomes at Map, highlighting Map in one color and all 'variable' proteins in one other color (instead of different colors depending on protein groups). The genomes were reordered to show LUZ19 on top, keeping groups defined on the basis of full proteome similarity together.  

Text was removed, and manually added again in powerpoint. 

In [ ]:
# code to determine the order of the phages 
df = pd.read_csv(os.path.join(pipeline_map_dir, "b_conservation", "lovis4u", "map_region", "proteome_similarity_matrix.tsv"), sep = "\t", index_col=0)
start_genome = "NC_010326.1" # LUZ19

ordered = [start_genome]
remaining = set(df.index) - {start_genome}

current = start_genome

while remaining:
    # sort remaining genomes by similarity to current
    similarities = df.loc[current, list(remaining)].sort_values(ascending = False)
    # pick the best unused one
    next_genome = similarities.index[0]
    # update lists
    ordered.append(next_genome)
    remaining.remove(next_genome)
    current = next_genome

# output result
print("Genome order:")
for g in ordered:
    print(g)


Genome order:
NC_010326.1
NC_022746.1
NC_017865.1
NC_011105.1
MW406976.1
NC_027375.1
MZ553931.1
NC_047852.1
MN615699.1
OP556577.1
MN615698.1
NC_048168.1
MZ687409.1
NC_012418.1
NC_047952.1
OL802210.1
MW406975.1
NC_022091.1
NC_011107.1
NC_005045.1
LC727700.1
OL754589.1
OR208619.1
ON857926.1
OK539824.1
MN692672.2
MN602045.1
MN901924.1
OQ412633.1
NC_028836.1
NC_026602.1
NC_047953.1
OP292288.1
OQ319930.1
NC_009935.1
MW117144.1
OQ230793.1
ON857935.1
ON857928.1
ON857933.1
NC_047967.1
